# State Objective Budget-Usage BC Warm Start v1

Goal: continue the anti-overfit state-objective line by training a diagnostic behavior-cloning warm-start actor on the fold-native teacher targets selected in `state_objective_fold_native_teacher_router_v1`.

External basis:
- Qlib keeps the project inside a reproducible quantitative investment workflow and artifact discipline: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL reinforces the train / validate / test separation for financial DRL experiments: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- Demonstration / behavior-cloning pretraining before RL improvement is a standard stabilizer in deep RL from demonstrations: https://arxiv.org/abs/1704.03732
- Overfit risk is treated as a first-class selection problem, consistent with the backtest-overfitting warning in Bailey et al.: https://doi.org/10.1090/conm/729/14680

Governance:
- This is **not** a promotion notebook. The source targets are derived from existing OOS / walk-forward reviews, so this actor can only be a diagnostic warm start and a candidate input to a future fold-local walk-forward PPO run.
- The acceptance target for the next real training run remains `relative_return >= +30pp`, `max_dd <= 15%`, plus walk-forward validation.
- Single-run logic and conclusion stay in this notebook; reusable training and audit helpers live in `src/soxl_quant/app/state_objective_bc.py`.


In [1]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import json
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "SPEC.md").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch

from src.soxl_quant.app.artifacts import persist_actor_bundle
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCTrainingConfig,
    evaluate_bc_actor,
    fit_state_objective_bc_actor_from_frame,
)
from src.soxl_quant.config import load_profile
from src.soxl_quant.policy.observation import observation_dim_for_profile
from src.soxl_quant.policy.policy_builder import build_state_objective_warm_start_dataset_from_frame

EXPERIMENT = "state_objective_budget_usage_bc_warm_start_v1"
CARD = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
SOURCE_CARD = ROOT / "runtime" / "model_selection" / "card" / "state_objective_fold_native_teacher_router_v1"
TARGETS_PATH = SOURCE_CARD / "fold_native_teacher_targets.csv"
SOURCE_SELECTION_PATH = SOURCE_CARD / "selection.json"
PROFILE_NAME = "risk_budget_usage_v1"
# `risk_budget_usage_v1`'s own cached frame ends in 2018. The v22 frame covers
# the 2020-2022 fold-native target dates, and the profile only consumes the
# default feature pack columns, so the extra vix-term columns are ignored.
FEATURE_FRAME_PROFILE = "balanced_rule_residual_v22"
FRAME_PATH = ROOT / "data" / f"soxl_profile_{FEATURE_FRAME_PROFILE}.pkl"
NATIVE_FRAME_PATH = ROOT / "data" / f"soxl_profile_{PROFILE_NAME}.pkl"

CARD.mkdir(parents=True, exist_ok=True)
profile = load_profile(PROFILE_NAME)
targets = pd.read_csv(TARGETS_PATH, parse_dates=["datetime"])
source_selection = json.loads(SOURCE_SELECTION_PATH.read_text())
frame = pd.read_pickle(FRAME_PATH)
native_frame = pd.read_pickle(NATIVE_FRAME_PATH)

train_config = StateObjectiveBCTrainingConfig(steps=600, lr=0.001, batch_size=128, seed=42)
print({
    "experiment": EXPERIMENT,
    "profile": PROFILE_NAME,
    "feature_frame_profile": FEATURE_FRAME_PROFILE,
    "target_rows": len(targets),
    "source_decision": source_selection.get("decision"),
    "source_training_allowed": source_selection.get("training_allowed"),
})

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'experiment': 'state_objective_budget_usage_bc_warm_start_v1', 'profile': 'risk_budget_usage_v1', 'feature_frame_profile': 'balanced_rule_residual_v22', 'target_rows': 1261, 'source_decision': 'fold_native_teacher_router_targets_ready', 'source_training_allowed': True}


## Source And Coverage Audit

The previous notebook selected `risk_budget_usage_v1` because its action surface was not label-degenerate. For this training pass the native cached frame cannot be used directly because it ends in 2018. The v22 feature frame is used only as a date-covered feature source; the `risk_budget_usage_v1` profile still determines observation columns, action interpreter, and actor architecture.


In [2]:
def date_index(df: pd.DataFrame) -> pd.DatetimeIndex:
    if "datetime" in df.columns:
        return pd.DatetimeIndex(pd.to_datetime(df["datetime"]).dt.normalize().unique())
    return pd.DatetimeIndex(pd.to_datetime(df.index).normalize().unique())

target_dates = pd.DatetimeIndex(pd.to_datetime(targets["datetime"]).dt.normalize().unique())
frame_dates = date_index(frame)
native_dates = date_index(native_frame)
overlap_dates = target_dates.intersection(frame_dates)
native_overlap_dates = target_dates.intersection(native_dates)

coverage = {
    "profile": PROFILE_NAME,
    "feature_frame_profile": FEATURE_FRAME_PROFILE,
    "frame_path": str(FRAME_PATH.relative_to(ROOT)),
    "native_frame_path": str(NATIVE_FRAME_PATH.relative_to(ROOT)),
    "targets_path": str(TARGETS_PATH.relative_to(ROOT)),
    "target_rows": int(len(targets)),
    "target_unique_dates": int(len(target_dates)),
    "target_start": str(target_dates.min().date()),
    "target_end": str(target_dates.max().date()),
    "feature_frame_rows": int(len(frame)),
    "feature_frame_start": str(frame_dates.min().date()),
    "feature_frame_end": str(frame_dates.max().date()),
    "feature_frame_overlap_dates": int(len(overlap_dates)),
    "native_frame_rows": int(len(native_frame)),
    "native_frame_start": str(native_dates.min().date()),
    "native_frame_end": str(native_dates.max().date()),
    "native_frame_overlap_dates": int(len(native_overlap_dates)),
    "source_training_allowed": bool(source_selection.get("training_allowed")),
    "source_promotion_allowed": bool(source_selection.get("promotion_allowed")),
    "source_teacher_scope": source_selection.get("teacher_source_scope"),
}
(CARD / "coverage.json").write_text(json.dumps(coverage, indent=2, sort_keys=True), encoding="utf-8")
coverage

{'profile': 'risk_budget_usage_v1',
 'feature_frame_profile': 'balanced_rule_residual_v22',
 'frame_path': 'data/soxl_profile_balanced_rule_residual_v22.pkl',
 'native_frame_path': 'data/soxl_profile_risk_budget_usage_v1.pkl',
 'targets_path': 'runtime/model_selection/card/state_objective_fold_native_teacher_router_v1/fold_native_teacher_targets.csv',
 'target_rows': 1261,
 'target_unique_dates': 756,
 'target_start': '2020-01-02',
 'target_end': '2022-12-30',
 'feature_frame_rows': 2973,
 'feature_frame_start': '2011-03-11',
 'feature_frame_end': '2022-12-30',
 'feature_frame_overlap_dates': 756,
 'native_frame_rows': 1965,
 'native_frame_start': '2011-03-11',
 'native_frame_end': '2018-12-31',
 'native_frame_overlap_dates': 0,
 'source_training_allowed': True,
 'source_promotion_allowed': False,
 'source_teacher_scope': 'fold_native_router_config_on_existing_base_upside_paths'}

## Train The Diagnostic BC Actor

The actor is trained only to reproduce the state-objective labels. This checks representational fit and produces a warm-start bundle for the next fold-local PPO experiment. It does not backtest or promote the actor by itself.


In [3]:
result = fit_state_objective_bc_actor_from_frame(
    frame,
    targets=targets,
    profile=profile,
    train_config=train_config,
    sample_size=0,
)

action_distribution = result.action_distribution.copy()
action_distribution.to_csv(CARD / "bc_action_distribution.csv", index=False)

bc_fit_metrics = dict(result.metrics)
bc_fit_metrics.update(
    {
        "obs_shape": [int(v) for v in result.obs.shape],
        "action_shape": [int(v) for v in result.actions.shape],
        "action_count": int(len(action_distribution)),
        "unique_actions": int((action_distribution["count"] > 0).sum()),
        "dominant_action": int(action_distribution.sort_values("count", ascending=False).iloc[0]["action"]),
        "dominant_action_share": float(action_distribution["share"].max()),
    }
)
(CARD / "bc_fit_metrics.json").write_text(json.dumps(bc_fit_metrics, indent=2, sort_keys=True), encoding="utf-8")

pd.DataFrame([bc_fit_metrics])

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


,samples,steps,lr,batch_size,before_loss,before_accuracy,train_last_loss,train_last_batch_accuracy,after_loss,after_accuracy,obs_shape,action_shape,action_count,unique_actions,dominant_action,dominant_action_share
0,755,600,0.001,128,1.614131,0.129801,1.064738,0.84375,1.07326,0.831788,"[755, 315]",[755],5,5,0,0.402649


In [4]:
action_distribution

,action,count,share
0,0,304,0.402649
1,1,297,0.393377
2,2,111,0.147020
3,3,4,0.005298
4,4,39,0.051656


## Fold And State Slice Audit

A high aggregate BC accuracy can still hide overfit to one fold or one state type. The following evaluation rebuilds labels by fold and by target state, then scores the same trained actor without updating weights.


In [5]:
def eval_subset(label: str, subset: pd.DataFrame) -> dict[str, object]:
    if subset.empty:
        return {"slice": label, "samples": 0, "loss": np.nan, "accuracy": np.nan, "unique_actions": 0, "dominant_action_share": np.nan}
    try:
        obs, actions = build_state_objective_warm_start_dataset_from_frame(
            frame,
            targets=subset,
            profile=profile,
            sample_size=0,
        )
    except RuntimeError:
        return {"slice": label, "samples": 0, "loss": np.nan, "accuracy": np.nan, "unique_actions": 0, "dominant_action_share": np.nan}
    metrics = evaluate_bc_actor(result.actor, obs, actions)
    counts = pd.Series(actions).value_counts(normalize=True)
    metrics.update(
        {
            "slice": label,
            "unique_actions": int(counts.shape[0]),
            "dominant_action_share": float(counts.max()) if len(counts) else np.nan,
        }
    )
    return metrics

fold_metrics = pd.DataFrame(
    [eval_subset(str(fold), subset) for fold, subset in targets.groupby("fold", dropna=False)]
).sort_values("slice")
fold_metrics.to_csv(CARD / "fold_bc_metrics.csv", index=False)
fold_metrics

,samples,loss,accuracy,slice,unique_actions,dominant_action_share
0,505,1.079593,0.825743,wf_2022_train,5,0.463366
1,755,1.077534,0.827815,wf_2023_2024_train,5,0.398675


In [6]:
state_slices = {
    "all": targets,
    "rebound_cluster": targets[targets["is_rebound_cluster"].astype(bool)],
    "downtrend_state": targets[targets["is_downtrend_state"].astype(bool)],
    "ordinary_state": targets[
        (~targets["is_rebound_cluster"].astype(bool))
        & (~targets["is_downtrend_state"].astype(bool))
    ],
}
state_metrics = pd.DataFrame([eval_subset(name, subset) for name, subset in state_slices.items()])
state_metrics.to_csv(CARD / "state_slice_bc_metrics.csv", index=False)
state_metrics

,samples,loss,accuracy,slice,unique_actions,dominant_action_share
0,755,1.073260,0.831788,all,5,0.402649
1,7,1.474499,0.428571,rebound_cluster,2,0.571429
2,477,1.090691,0.813417,downtrend_state,3,0.505241
3,278,1.174270,0.723022,ordinary_state,5,0.500000


## Persist Diagnostic Actor Bundle

The bundle is tagged, so it does not replace the canonical `risk_budget_usage_v1` actor. It is retained as a local/trackable warm-start artifact for the next walk-forward PPO experiment.


In [7]:
actor_path = persist_actor_bundle(
    profile=profile,
    actor=result.actor,
    obs_dim=observation_dim_for_profile(profile),
    seed=int(train_config.seed),
    effective_tag=EXPERIMENT,
)
actor_record = {
    "actor_path": str(actor_path.relative_to(ROOT)),
    "profile": PROFILE_NAME,
    "tag": EXPERIMENT,
    "seed": int(train_config.seed),
    "obs_dim": int(observation_dim_for_profile(profile)),
}
(CARD / "actor_record.json").write_text(json.dumps(actor_record, indent=2, sort_keys=True), encoding="utf-8")
actor_record

{'actor_path': 'runtime/model/card/risk_budget_usage_v1/soxl_risk_budget_usage_v1_state_objective_budget_usage_bc_warm_start_v1_actor.pt',
 'profile': 'risk_budget_usage_v1',
 'tag': 'state_objective_budget_usage_bc_warm_start_v1',
 'seed': 42,
 'obs_dim': 315}

## Selection Decision

This notebook can only decide whether the BC actor is a usable warm-start artifact. Promotion remains blocked until a future fold-local PPO run produces fixed-split and walk-forward backtests meeting the `+30pp / <=15% MaxDD` target.


In [8]:
min_fold_accuracy = float(fold_metrics["accuracy"].dropna().min()) if not fold_metrics.empty else float("nan")
state_non_all = state_metrics[state_metrics["slice"] != "all"].copy()
min_state_accuracy = float(state_non_all["accuracy"].dropna().min())
rebound_row = state_metrics.loc[state_metrics["slice"] == "rebound_cluster"].iloc[0]
rebound_samples = int(rebound_row["samples"])
rebound_accuracy = float(rebound_row["accuracy"])

bc_fit_gate = bool(bc_fit_metrics["after_accuracy"] >= 0.80)
fold_consistency_gate = bool(min_fold_accuracy >= 0.80)
label_distribution_gate = bool(
    bc_fit_metrics["unique_actions"] >= 3
    and bc_fit_metrics["dominant_action_share"] <= 0.90
)
rebound_coverage_gate = bool(rebound_samples >= int(source_selection.get("min_train_rebound_days", 3)))
state_slice_fit_gate = bool(min_state_accuracy >= 0.60 and rebound_accuracy >= 0.60)
global_bc_fit_ready = bool(
    source_selection.get("training_allowed")
    and bc_fit_gate
    and fold_consistency_gate
    and label_distribution_gate
    and rebound_coverage_gate
)
warm_start_ready = bool(global_bc_fit_ready and state_slice_fit_gate)

if warm_start_ready:
    decision = "diagnostic_bc_warm_start_ready_for_walkforward_ppo"
elif global_bc_fit_ready and not state_slice_fit_gate:
    decision = "diagnostic_bc_global_fit_rebound_slice_weak_no_promotion"
else:
    decision = "diagnostic_bc_warm_start_not_ready"

selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "profile": PROFILE_NAME,
    "feature_frame_profile": FEATURE_FRAME_PROFILE,
    "source_experiment": "state_objective_fold_native_teacher_router_v1",
    "source_decision": source_selection.get("decision"),
    "promotion_allowed": False,
    "promotion_blocker": "BC label fit is not a backtest; actor uses targets derived from previous review evidence and must enter fold-local walk-forward PPO plus fixed-split gates before promotion.",
    "global_bc_fit_ready": global_bc_fit_ready,
    "warm_start_ready_for_next_training": warm_start_ready,
    "next_training_requires_rebound_weighting": bool(global_bc_fit_ready and not state_slice_fit_gate),
    "next_mechanism": "rebound_weighted_state_objective_bc_then_walkforward_ppo" if not state_slice_fit_gate else "walkforward_ppo_from_state_objective_budget_usage_bc_actor",
    "acceptance_gate_for_next_run": {
        "relative_return_min_pp": 30.0,
        "max_dd_max_pct": 15.0,
        "walkforward_required": True,
        "fixed_split_required": True,
        "rebound_slice_accuracy_min": 0.60,
        "no_runtime_promotion_from_bc_only": True,
    },
    "bc_fit_gate": bc_fit_gate,
    "fold_consistency_gate": fold_consistency_gate,
    "label_distribution_gate": label_distribution_gate,
    "rebound_coverage_gate": rebound_coverage_gate,
    "state_slice_fit_gate": state_slice_fit_gate,
    "after_accuracy": float(bc_fit_metrics["after_accuracy"]),
    "after_loss": float(bc_fit_metrics["after_loss"]),
    "min_fold_accuracy": min_fold_accuracy,
    "min_state_accuracy": min_state_accuracy,
    "rebound_accuracy": rebound_accuracy,
    "rebound_samples": rebound_samples,
    "samples": int(bc_fit_metrics["samples"]),
    "unique_actions": int(bc_fit_metrics["unique_actions"]),
    "dominant_action_share": float(bc_fit_metrics["dominant_action_share"]),
    "actor_path": actor_record["actor_path"],
    "external_sources": [
        "https://github.com/microsoft/qlib",
        "https://arxiv.org/abs/2009.11189",
        "https://github.com/AI4Finance-Foundation/FinRL",
        "https://arxiv.org/abs/2011.09607",
        "https://arxiv.org/abs/1704.03732",
        "https://doi.org/10.1090/conm/729/14680",
    ],
    "sidecars": [
        "coverage.json",
        "bc_fit_metrics.json",
        "bc_action_distribution.csv",
        "fold_bc_metrics.csv",
        "state_slice_bc_metrics.csv",
        "actor_record.json",
        "summary.json",
    ],
}
summary = {
    "conclusion": [
        f"BC fit improved from {bc_fit_metrics['before_accuracy']:.3f} to {bc_fit_metrics['after_accuracy']:.3f} accuracy on {bc_fit_metrics['samples']} samples.",
        f"Fold consistency gate {'passed' if fold_consistency_gate else 'failed'} with min fold accuracy {min_fold_accuracy:.3f}.",
        f"Label distribution gate {'passed' if label_distribution_gate else 'failed'} with {bc_fit_metrics['unique_actions']} actions and dominant share {bc_fit_metrics['dominant_action_share']:.3f}.",
        f"Rebound slice fit gate {'passed' if state_slice_fit_gate else 'failed'}: rebound accuracy {rebound_accuracy:.3f} on {rebound_samples} samples, so the next run must use rebound-weighted BC or sampling before PPO.",
        "This is a warm-start artifact only; no runtime promotion without fold-local PPO, fixed-split backtest, and walk-forward validation.",
    ],
    "coverage": coverage,
    "bc_fit_metrics": bc_fit_metrics,
    "fold_metrics": fold_metrics.to_dict("records"),
    "state_metrics": state_metrics.to_dict("records"),
    "actor_record": actor_record,
    "selection": selection,
}
(CARD / "selection.json").write_text(json.dumps(selection, indent=2, sort_keys=True), encoding="utf-8")
(CARD / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
selection

{'experiment': 'state_objective_budget_usage_bc_warm_start_v1',
 'created_at': '2026-05-25T11:53:08.278499+00:00',
 'decision': 'diagnostic_bc_global_fit_rebound_slice_weak_no_promotion',
 'profile': 'risk_budget_usage_v1',
 'feature_frame_profile': 'balanced_rule_residual_v22',
 'source_experiment': 'state_objective_fold_native_teacher_router_v1',
 'source_decision': 'fold_native_teacher_router_targets_ready',
 'promotion_allowed': False,
 'promotion_blocker': 'BC label fit is not a backtest; actor uses targets derived from previous review evidence and must enter fold-local walk-forward PPO plus fixed-split gates before promotion.',
 'global_bc_fit_ready': True,
 'warm_start_ready_for_next_training': False,
 'next_training_requires_rebound_weighting': True,
 'next_mechanism': 'rebound_weighted_state_objective_bc_then_walkforward_ppo',
 'acceptance_gate_for_next_run': {'relative_return_min_pp': 30.0,
  'max_dd_max_pct': 15.0,
  'walkforward_required': True,
  'fixed_split_required': Tr